# 3. Case 2s: look-alike street swaps (the paper's dataset)

Real case 2 pairs (same house number, similar but different street) are too few to evaluate on (15 in test). This notebook builds many more from real positives: X = a place, Y = a place on a different but similar-looking street (Jaro-Winkler > 0.8) in the same ZIP; the new pair is X's Overture address vs **X's house number + Y's Foursquare street text**, label 0, `neg_type = "case2s"`. Swaps **replace** the same number of easy negatives per split, so every split stays 1:9.

Reads the pairs written by `02_build_pairs.ipynb`; writes `Data/ct_pairs_{split}_case2s.parquet`, which every model reads through `load_pairs("overture_case2s", split)`. Then run `04_case2s_checks.ipynb`.

| § | Stage | Output variable |
|---|---|---|
| 0 | Setup | |
| — | Load the final pairs | `pairs` |
| 1 | Load places | `places` |
| 2 | Clean house numbers only | `places` |
| 4 | Unique streets per (split, ZIP), X side and Y side | `streets_a`, `streets_b` |
| 5 | Street × street pairs within a ZIP | `street_pairs` |
| 6 | Flag same street, written differently (libpostal) | `street_pairs` |
| 7 | Flag directionally incompatible pairs | `street_pairs` (`dir_disq`) |
| 8 | Look-alike: Jaro-Winkler | `street_pairs` |
| 9 | `swap_type` and the kept pairs (JW > 0.8, no flag) | `street_pairs`, `kept` |
| 10 | Back to places (X, Y) | `swaps` |
| 11 | Build the pairs | `swaps` |
| 12 | Counts → decide the cap | |
| 13 | Cap at 5 rows per street pair, replace easy negatives, write | new pair files |

## 0. Setup
**Why:** one place for the imports and the constants every later cell uses (`SEED` for every shuffle and sample, so a re-run gives the same files).

In [ ]:
# §0: imports and constants
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")   # repository root -> lets us import helpers/ and the model packages

import numpy as np
import pandas as pd
import jellyfish
from postal.expand import expand_address

from helpers.pairs import load_pairs, add_number_street
from helpers.check_model_performance import find_random_sample_given_mask

OVERTURE_RELEASE = "2026-09-23.1"   # must match 01 / 02
SEED = 0
pd.set_option("display.width", 250)              # widen the line
pd.set_option("display.expand_frame_repr", False)  # never wrap, scroll sideways

## Load the final pairs
**Why:** the swaps are added to the finished Overture pair set from `02`, so everything starts there: it fixes which ZIPs are train / val / test, and it is the set the swaps will replace rows in.  
**Does:** reads the three splits, puts them in one frame, re-attaches the `label` column. Nothing is filtered yet.  
**Output:** `pairs` (the set before any swap)

In [ ]:
# Load: all splits -> `pairs` (also the lookup for the "not already in any split" acceptance check)
splits = []
for split in ["train", "val", "test"]:
    df, label = load_pairs("overture", split)
    splits.append(df.assign(label=label))
pairs = pd.concat(splits, ignore_index=True)

print(pairs.shape)
pairs.groupby(["split", "neg_type"], observed=True).size().unstack()

## 1. Load places
**Why:** a swap needs a real place to take the house number from (X) and a real place to take a different street from (Y). Only positives (`case0`) are safe for that: in a positive, the Overture address and the Foursquare address are known to be the same place, so `street_a` and `street_b` both belong to one real place in one real ZIP. A negative is two different places, so it can't tell us "this place is on this street".  
**Does:** keep the `case0` rows, then split each address into house number + street with `add_number_street` (→ `number_a/b`, `street_a/b`). One row = one place.  
**Input → output:** `pairs` → `places`

In [ ]:
# §1: Load places
places = pairs[pairs["neg_type"] == "case0"]
places = add_number_street(places)

## 2. Clean house numbers only
**Why:** the new B address is "X's house number + Y's street text". That only reads as a normal address when the number is a plain number; `16-18`, `20A` or a missing number would give odd strings and make the "same number" check ambiguous.  
**Does:** keep places where `number_a` is only digits (`str.fullmatch(r"\d+")`).  
**Input → output:** `places` → `places`

In [ ]:
# §2: Clean house numbers only

mask = (places["number_a"].str.fullmatch(r"\d+", na = False))
places = places[mask]

## 4. Unique streets
**Why:** street pairs are decided between *streets*, not places. Hundreds of places share one street, so comparing places would repeat the same street comparison many times. Reduce to the unique streets first. (§3, "the two sources agree on the street", was dropped: no clean definition of "agree" without a street matcher, which is the task being trained.)  
**Does:** two small tables of unique streets, with the ZIP column renamed to `zip` and `split` kept (so a street never meets a street from another split). X side = Overture's street: `(split, zip_a, street_a)` → `streets_a`. Y side = Foursquare's street text: `(split, zip_b, street_b)` → `streets_b`.  
**Input → output:** `places` → `streets_a`, `streets_b`

In [ ]:
# §4: Unique streets

streets_a = places[["split", "zip_a", "street_a"]].drop_duplicates().rename(columns = {"zip_a": "zip"})

streets_b = places[["split", "zip_b", "street_b"]].drop_duplicates().rename(columns = {"zip_b": "zip"})

## 5. Street pairs
**Why:** a case2 negative needs two *different* streets in the same area (same ZIP), which is what makes it a realistic confusion. Joining every X street with every Y street of the same split and ZIP gives every candidate.  
**Does:** merge `streets_a` with `streets_b` on `(split, zip)`; drop pairs whose text is identical (`street_a != street_b`).  
**Input → output:** `streets_a` + `streets_b` → `street_pairs`

In [ ]:
# §5: Street pairs
street_pairs = streets_a.merge(streets_b, on = ["split", "zip"])
street_pairs = street_pairs[street_pairs["street_a"] != street_pairs["street_b"]]
street_pairs.head()

## 6. Same street, written differently (flags)
**Why:** two different strings can still be the same street (`MAIN ST` vs `MAIN STREET`, `FARMINGTON AVE` vs `FARMINGTON AVE STE 2`). Labelled 0, that would be a true match with a wrong label. This step marks those pairs; the rows stay, and the flags are applied together in §9.  
**Does:** two flag columns. `lib_match`: libpostal's expansions of the two streets share a string (`expand_address`, run once per unique street through a dict, not once per pair). `prefix_match`: one expansion starts with the other plus a space (extra text after the street).  
**Input → output:** `street_pairs` → `street_pairs` (adds `street_a_exp`, `street_b_exp`, `lib_match`, `prefix_match`)

In [ ]:
# §6: Drop same street, written differently
import itertools


unique_names = set(street_pairs["street_a"].tolist() + street_pairs["street_b"].tolist())
lookup = {name: set(expand_address(name, languages= ["en"])) for name in unique_names}

street_pairs["street_a_exp"] = street_pairs["street_a"].map(lookup)
street_pairs["street_b_exp"] = street_pairs["street_b"].map(lookup)
street_pairs["lib_match"] = pd.Series(
    [len(a & b) > 0 for a,b in zip(street_pairs["street_a_exp"], street_pairs["street_b_exp"])],
    index = street_pairs.index
)

def compare_prefix_sets(a, b):
    for ax, bx in itertools.product(a, b):
        ax, bx = ax + " ", bx + " "
        if ax.startswith(bx) or bx.startswith(ax):
            return True
    return False

street_pairs["prefix_match"] = pd.Series(
    [compare_prefix_sets(a,b) for a,b in zip(street_pairs["street_a_exp"], street_pairs["street_b_exp"])],
    index = street_pairs.index
)

for prefix_match in [True, False]:
    print(street_pairs[(street_pairs["prefix_match"] == prefix_match) & ~street_pairs["lib_match"]].sample(frac =1).head(5)[["street_a", "street_b", "prefix_match"]])

## 7. Flag directionally incompatible pairs
**Why:** `N MAIN ST` vs `MAIN ST` might be the same street or two different ones, and we can't tell, so we don't use that pair as a negative. `N MAIN ST` vs `S MAIN ST` are clearly different streets and make good hard negatives, so they stay.  
**Does:** extract the direction word (whole words only: `N S E W NE NW SE SW NORTH SOUTH EAST WEST NORTHEAST NORTHWEST SOUTHEAST SOUTHWEST`) from `street_a` and `street_b`. Flag the pair (`dir_disq`) when **exactly one side** has one. The no-direction versions (`street_a_nod`, `street_b_nod`) are kept for a later idea (coordinate clustering per street). Rows stay; flags are applied together in §9.  
**Input → output:** `street_pairs` → `street_pairs` (adds `street_a_d`, `street_b_d`, `street_a_nod`, `street_b_nod`, `dir_disq`)

In [ ]:
# §7: Drop directional-only
DIRECTIONALS = {
    "N", "S", "E", "W", "NE", "NW", "SE", "SW",
    "NORTH", "SOUTH", "EAST", "WEST",
    "NORTHEAST", "NORTHWEST", "SOUTHEAST", "SOUTHWEST",
}

extract_dir = lambda x: x.str.extract(rf"(\b(?:{'|'.join(DIRECTIONALS)})\b)")
fully_replace_dir = lambda x: remove_white(replace_dir(x))

replace_dir = lambda x: x.str.replace(rf"\b(?:{'|'.join(DIRECTIONALS)})\b", "", regex = True)
remove_white = lambda x: x.str.replace(r"\s+"," ", regex = True).str.strip()

for label in ["a", "b"]:
    street_pairs[f"street_{label}_d"] = extract_dir(street_pairs[f"street_{label}"])
    street_pairs[f"street_{label}_nod"] = fully_replace_dir(street_pairs[f"street_{label}"])



diff = (
    (street_pairs["street_a_d"].isna() & (~street_pairs["street_b_d"].isna())) |
    (street_pairs["street_b_d"].isna() & (~street_pairs["street_a_d"].isna()))
)
street_pairs["dir_disq"] = diff 


print(street_pairs[diff]) #72343 row dropped


# print(street_pairs[
#     (street_pairs["street_a_nod"] == street_pairs["street_b_nod"] ) &
#     (street_pairs["street_a"] != street_pairs["street_b"]) 
# ])

## 8. Look-alike (Jaro-Winkler)
**Why:** a case2 negative should be hard: two streets that look alike to a string matcher, like real case2 (Jaro-Winkler > 0.8). Jaro-Winkler is a 0–1 score of how alike two strings are.  
**Does:** compute `jw_similarity` for every street pair (whole string, suffix included). The loop prints, per threshold from 0.70, how many unflagged pairs survive, and two sample pairs near each threshold, to judge where the cut should be. No rows are removed here; the 0.8 cut is applied in §9.  
**Input → output:** `street_pairs` → `street_pairs` (adds `jw_similarity`)

In [ ]:
# §8: Look-alike
street_pairs["jw_similarity"] = pd.Series(
    [jellyfish.jaro_winkler_similarity(a,b) for a,b in zip(street_pairs["street_a"], street_pairs["street_b"])],
    index = street_pairs.index
)
find_jw_thresh = lambda x, thresh: x[x["jw_similarity"] >= thresh].sample(frac = 1)
filter_not_disq = lambda x: x[(x["lib_match"] == False) & (x["prefix_match"] == False) & (x["dir_disq"] == False)]

MARGIN = 0.01
lst = []
freq_list = {}
for thresh in np.arange(0.70, 1, 0.05):
    left, right = thresh - MARGIN, thresh + MARGIN
    res = find_jw_thresh(filter_not_disq(street_pairs), thresh)
    # print(f"Thresh: {thresh:.2f} | N: {len(res)}")
    freq_list[f"{thresh:.2f}"] = len(res)
    lst.append(
        res[(left<res["jw_similarity"]) & 
        (res["jw_similarity"] < right)]
        [["street_a", "street_b", "jw_similarity"]]
        .head(2)
    )

print(pd.DataFrame.from_dict(freq_list, orient = "index", columns = ["N"]))
print(pd.concat(lst))

## 9. `swap_type` and the kept pairs
**Why:** there are two kinds of look-alike: only the suffix differs (`ELM ST` vs `ELM AVE`), or the name itself differs. Both are labelled 0 but teach different things, so each pair gets a type and results can be reported per type. This is also the step that turns the flags into one filter.  
**Does:** `swap_type = suffix_only` if every word but the last is equal, else `name_differs`. Then `kept` = the street pairs with no flag (`~lib_match`, `~prefix_match`, `~dir_disq`) and `jw_similarity > 0.8`.  
**Input → output:** `street_pairs` → `street_pairs` (adds `swap_type`), `kept`

In [ ]:
# §9: swap_type
extract_non_suffix = lambda x: x.str.replace(r"(\s\b\w+\b)$", "", regex = True)


street_pairs["swap_type"] = (extract_non_suffix(street_pairs["street_a"]) == extract_non_suffix(street_pairs["street_b"])).map(
    { True: "suffix_only", False: "name_differs"})

# for swap_type in ["suffix_only", "name_differs"]:
#     print(street_pairs[["street_a", "street_b", "swap_type"]][street_pairs["swap_type"] == swap_type].sample(frac=1).head(5))

kept = street_pairs[(~street_pairs["lib_match"]) & (~street_pairs["prefix_match"]) & (~street_pairs["dir_disq"]) & (street_pairs["jw_similarity"] > 0.8)]

kept["swap_type"].value_counts()
print(kept)
print(kept["swap_type"].value_counts())

### 9b. How many swaps would this give?
**Why:** each kept street pair becomes one swap per X place on `street_a`. Before building anything, estimate the size, to see whether the threshold gives enough rows.  
**Does:** count the places per `(split, zip_a, street_a)` (`n_x`), join the count onto `kept`; the sum of `n_x` is the number of rows §10 starts from. The commented lines are a histogram of `n_x`.  
**Input → output:** `kept` + `places` → `new_kept` (only printed)

In [ ]:
n_x = places.groupby(["split", "zip_a", "street_a"]).size().rename("n_x").reset_index()

new_kept = kept.merge(
    n_x, 
    left_on = ["split", "zip", "street_a"], 
    right_on = ["split", "zip_a", "street_a"], 
    how = "left"
    )
print(new_kept.head(1))
print(new_kept["n_x"].sum())
# print(new_kept[["street_a", "street_b", "n_x"]].sort_values(by = "n_x", ascending = False))


# import matplotlib.pyplot as plt
# #X-axis = unique_id
# # Y-Axis = frequency
# plt.hist(new_kept["n_x"], bins = 20)
# plt.yscale("log")
# plt.ylabel('n')
# plt.xlabel("unique street combinations")
# plt.show()

## 10. Back to places
**Why:** a street pair is not a pair of addresses yet. A swap needs a real X (an address with a house number) and the street text of a real Y.  
**Does:** X side: `x_rows` = every place on `street_a` in that ZIP (one row per X). Y side: `ys` = one place per `(split, zip_b, street_b)`; it only supplies the Foursquare street text and an id. Join them on `(split, zip, street_b)`: `split` is in every key so Y never comes from another split. Last, `swaps[id_a != id_b]` removes rows where Y is X itself (a place whose own Foursquare text differs from its Overture street by a typo would otherwise be labelled "different").  
**Input → output:** `kept` + `places` → `swaps`

In [ ]:
# §10: Back to places

# X side: every place on street_a (same ZIP) -> one row per X place
x_rows = kept.merge(
    places[["zip_a", "street_a", "id_a", "address_a", "number_a"]],
    left_on = ["zip", "street_a"], right_on = ["zip_a", "street_a"],
)
swaps = x_rows.merge(
    places[["split","zip_b", "street_b", "id_b"]].drop_duplicates(["split", "zip_b", "street_b"]),
    left_on = ["split","zip", "street_b"], right_on = ["split","zip_b", "street_b"],
)
swaps = swaps[swaps["id_a"] != swaps["id_b"]]

## 11. Build the pairs
**Why:** the output has to look like every other pair file (same columns) so the models can read it. The new B address uses X's house number, so only the street differs: same number, different street, which is what case2 means.  
**Does:** `address_a` = X's Overture address · `address_b` = X's number + " " + Y's `street_b` · `zip_a` = X's · `zip_b` = Y's · `id_a` = X · `id_b` = Y · `split` · label 0 · `neg_type = "case2s"` · keep `jw`, `swap_type`, and add `street_a` (X's) and `street_b` (Y's, the street in the new B). Last, `drop_duplicates(["address_a", "address_b"])`: several places share one Overture address and would otherwise give identical rows.  
**Input → output:** `swaps` → `swaps` (shared schema + `jw`, `swap_type`, `street_a`, `street_b`)

In [ ]:
# §11: Build the pairs (run §10 first: this renames columns in place)
swaps["address_b"] = swaps["number_a"] + " " + swaps["street_b"]
swaps["number_b"] = swaps["number_a"]
swaps["label"] = 0
swaps["neg_type"] = "case2s"

swaps = swaps.rename(columns = {"jw_similarity": "jw"})[
    ["id_a", "id_b", "address_a", "address_b", "zip_a", "zip_b", "split", "label", "neg_type", 
     "number_a", "number_b", "street_a", "street_b", "jw", "swap_type"]
]
swaps = swaps.drop_duplicates(["address_a", "address_b"])

## 12. Counts → decide the cap
**Why:** before writing anything, check how many swaps each split has and whether a few street pairs supply most of them, so the cap in §13 is chosen from numbers.  
**Does:** per split: rows, distinct X places, street pairs, median and max rows per pair, share of the top 10 pairs. Then the number of rows left if each street pair is capped at 3, 5 or 10 rows (`clip(upper=k)`; dropping whole pairs instead would lose far more).  
**Input:** `swaps`

In [ ]:
# §12: counts per split (one row per split)
summary = []
for split, sub in swaps.groupby("split", observed = True):
    by_pair = sub.groupby(["street_a", "street_b"]).size().sort_values(ascending = False)
    summary.append({
        "split": split,
        "rows": len(sub),
        "x_places": sub["id_a"].nunique(),
        "street_pairs": len(by_pair),
        "median_per_pair": by_pair.median(),
        "max_per_pair": by_pair.max(),
        "top10_share_%": round(by_pair.head(10).sum() * 100 / by_pair.sum(), 1),
    })

display(pd.DataFrame(summary).set_index("split"))

summary = []
for split, sub in swaps.groupby("split", observed=True):
    by_pair = sub.groupby(["street_a", "street_b"]).size()
    n = len(sub)
    summary.append(
        {
            "split": split,
            "n": n,
            **{
                f"<{kx}": by_pair.clip(upper=kx).sum()
                for kx in (3,5,10)
            },
            **{
                f"<{kx} pct": f"{by_pair.clip(upper=kx).sum()/n:.0%}"
                for kx in (3,5,10)
            }
        }
    )
display(pd.DataFrame(summary).set_index("split"))

## 13. Cap, replace easy negatives, write
**Why:** the swaps must *replace* easy negatives, not be added on top, so every split stays 1:9; and a cap on rows per street pair stops a few pairs dominating the new slice. The result is written as a separate dataset so the original files stay untouched.  
**Does:** shuffle with `SEED`, keep at most `K` rows per `(split, street_a, street_b)`. Per split: if there are more swaps than `case3` rows, sample the swaps down; drop that many random `case3` rows; append the swaps; write `Data/ct_pairs_{split}_case2s.parquet` (same 9 columns as the other pair files). `load_pairs("overture_case2s", split)` reads them.  
**Input → output:** `pairs` + `swaps` → `ct_pairs_{train,val,test}_case2s.parquet`

In [ ]:
# §13: sample swaps, drop the same number of case3 per split, write files
K = 5
capped = swaps.sample(frac = 1, random_state = SEED).groupby(["split", "street_a", "street_b"]).head(K)
for split in ["train", "val", "test"]:
    sub = pairs[pairs["split"] == split]
    new = capped[capped["split"] == split]
    n_easy = (sub["neg_type"] == "case3").sum()
    if len(new) > n_easy:
        new = new.sample(n = n_easy, random_state = SEED)


    drop_idx = sub[sub["neg_type"] == "case3"].sample(n = len(new), random_state = SEED).index


    out = pd.concat([sub.drop(index = drop_idx), new[sub.columns]], ignore_index = True)

    # display(out.tail(5))
    out.to_parquet(f"Data/ct_pairs_{split}_case2s.parquet", index = False)